# TP2 — Tester l'API : pytest, TestClient, OpenAPI
## Fil rouge : Churn Predictor (Session 4)

**Durée estimée : 50 min**

### 🎯 Objectifs d'apprentissage
À la fin de ce TP vous saurez :
- Utiliser **`TestClient` de FastAPI** pour tester l'API sans serveur live.
- Écrire des **tests pytest** : cas valides, cas invalides (422), erreurs métier (500).
- Lire et comprendre le **schéma OpenAPI** auto-généré.
- Vérifier la **discipline de versioning** d'une API.

### 📦 Pré-requis
- Avoir terminé le TP1 (API qui démarre, Swagger accessible).
- Ce notebook est exécuté **à la racine** du projet `churn_api/`.


In [1]:
# Setup Colab : dézipper le projet et installer les dépendances
!unzip -q churn_api_for_colab.zip -d .
!pip install -q fastapi "uvicorn[standard]" "pydantic>=2.5" httpx pytest "scikit-learn==1.6.1"

# Vérifier que la structure est OK
!ls -la
!echo "---"
!ls -la app/
!echo "---"
!ls -la artifacts/

total 216
drwxr-xr-x 1 root root   4096 Jun 12 12:09 .
drwxr-xr-x 1 root root   4096 Jun 12 11:44 ..
drwxr-xr-x 3 root root   4096 Jun 12 11:56 app
drwxr-xr-x 2 root root   4096 May 10 14:58 artifacts
-rw-r--r-- 1 root root 184509 Jun 12 12:09 churn_api_for_colab.zip
drwxr-xr-x 4 root root   4096 Jun  4 13:32 .config
-rw-r--r-- 1 root root    146 May 10 14:58 requirements.txt
drwxr-xr-x 1 root root   4096 Jun  4 13:32 sample_data
drwxr-xr-x 3 root root   4096 Jun 12 12:06 tests
---
total 32
drwxr-xr-x 3 root root 4096 Jun 12 11:56 .
drwxr-xr-x 1 root root 4096 Jun 12 12:09 ..
-rw-r--r-- 1 root root  445 May 10 14:58 config.py
-rw-r--r-- 1 root root   35 May 10 14:58 __init__.py
-rw-r--r-- 1 root root 2624 Jun 12 12:03 main.py
-rw-r--r-- 1 root root 2281 Jun 12 11:56 model.py
drwxr-xr-x 2 root root 4096 Jun 12 12:03 __pycache__
-rw-r--r-- 1 root root 3013 Jun 12 11:55 schemas.py
---
total 524
drwxr-xr-x 2 root root   4096 May 10 14:58 .
drwxr-xr-x 1 root root   4096 Jun 12 12:09 ..
-rw-

## 1. `TestClient` — tester sans serveur

🧠 **Le concept** : `TestClient` simule des requêtes HTTP sans démarrer de serveur. Il appelle directement l'app FastAPI, déclenche le `lifespan` (chargement modèle), et renvoie des objets `Response` comme si c'était un vrai HTTP.

**Avantages** :
- Aucun port à ouvrir, aucune dépendance réseau.
- Tests reproductibles, isolés, rapides.
- Intégré dans pytest naturellement.
- C'est ce qui tourne dans la CI/CD.

In [2]:
import sys
from pathlib import Path

# Make sure the local 'app' package is importable
sys.path.insert(0, str(Path.cwd()))

from fastapi.testclient import TestClient
from app.main import app

# IMPORTANT: TestClient as a context manager triggers the lifespan handler
# (without it, app.state.model_service would not be initialized)
client = TestClient(app)
client.__enter__()  # we'll close at the end

print("TestClient ready.")


✅ Model loaded: HGBT_tuned
TestClient ready.


## 2. Premier test : `/health`

C'est l'endpoint le plus simple — un bon début pour valider que tout est câblé.

In [3]:
response = client.get("/health")
print(f"Status: {response.status_code}")
print(f"Body: {response.json()}")

assert response.status_code == 200
assert response.json()["status"] == "ok"
assert response.json()["model_loaded"] is True
print("✅ /health OK")


Status: 200
Body: {'status': 'ok', 'model_loaded': True}
✅ /health OK


## 3. Test du `/predict` avec un client réel

On va construire un payload complet et l'envoyer.

In [4]:
SAMPLE_CLIENT = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 80.0,
    "TotalCharges": 960.0,
}

response = client.post("/predict", json=SAMPLE_CLIENT)
print(f"Status: {response.status_code}")
print(f"Body: {response.json()}")

assert response.status_code == 200
body = response.json()
assert 0 <= body["churn_probability"] <= 1
assert body["churn_predicted"] in (0, 1)
print("✅ /predict OK")


Status: 200
Body: {'churn_probability': 0.6972770249517803, 'churn_predicted': 1, 'threshold_used': 0.15000000000000002}
✅ /predict OK


## 4. Tester la validation Pydantic — les erreurs 422

🧠 **Pédagogiquement crucial** : on va envoyer **volontairement** des payloads invalides pour vérifier que l'API rejette proprement avec un 422 (et pas un 500 — ça veut dire qu'on a oublié une validation).

In [5]:
# Cas 1 : champ manquant
bad = SAMPLE_CLIENT.copy()
del bad["tenure"]

response = client.post("/predict", json=bad)
print(f"Status: {response.status_code}")
print(f"Errors: {response.json()['detail'][:1]}")

assert response.status_code == 422
print("✅ Missing field → 422")


Status: 422
Errors: [{'type': 'missing', 'loc': ['body', 'tenure'], 'msg': 'Field required', 'input': {'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'No', 'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No', 'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check', 'MonthlyCharges': 80.0, 'TotalCharges': 960.0}}]
✅ Missing field → 422


In [6]:
# Cas 2 : valeur hors de l'enum Literal
bad = SAMPLE_CLIENT.copy()
bad["Contract"] = "Five years"  # not in Literal

response = client.post("/predict", json=bad)
print(f"Status: {response.status_code}")
print(f"Error message: {response.json()['detail'][0]['msg']}")

assert response.status_code == 422
print("✅ Invalid enum → 422")


Status: 422
Error message: Input should be 'Month-to-month', 'One year' or 'Two year'
✅ Invalid enum → 422


In [7]:
# Cas 3 : valeur hors de Field(ge=0, le=120)
bad = SAMPLE_CLIENT.copy()
bad["tenure"] = -5

response = client.post("/predict", json=bad)
assert response.status_code == 422
print(f"Error: {response.json()['detail'][0]['msg']}")
print("✅ Out-of-bounds → 422")


Error: Input should be greater than or equal to 0
✅ Out-of-bounds → 422


In [8]:
# Cas 4 : mauvais type
bad = SAMPLE_CLIENT.copy()
bad["MonthlyCharges"] = "not a number"

response = client.post("/predict", json=bad)
assert response.status_code == 422
print(f"Error: {response.json()['detail'][0]['msg']}")
print("✅ Wrong type → 422")


Error: Input should be a valid number, unable to parse string as a number
✅ Wrong type → 422


## 5. Test du batch endpoint

In [9]:
payload = {"clients": [SAMPLE_CLIENT, SAMPLE_CLIENT, SAMPLE_CLIENT]}
response = client.post("/predict-batch", json=payload)

assert response.status_code == 200
predictions = response.json()["predictions"]
print(f"Got {len(predictions)} predictions")
print(f"First: {predictions[0]}")
assert len(predictions) == 3
print("✅ /predict-batch OK")


Got 3 predictions
First: {'churn_probability': 0.6972770249517803, 'churn_predicted': 1, 'threshold_used': 0.15000000000000002}
✅ /predict-batch OK


In [10]:
# Test: empty batch must be rejected (BatchPredictionRequest has min_length=1)
response = client.post("/predict-batch", json={"clients": []})
assert response.status_code == 422
print("✅ Empty batch → 422 (rejected by Pydantic)")


✅ Empty batch → 422 (rejected by Pydantic)


## 6. Sous le capot : le schéma OpenAPI

🧠 **Le killer feature de FastAPI** : il génère automatiquement un schéma OpenAPI 3.x complet, exposé sur `/openapi.json` et rendu dans `/docs` (Swagger) et `/redoc` (ReDoc).

Ce schéma est **machine-readable** : on peut générer des **clients SDK** dans n'importe quel langage (TypeScript, Java, Go…) à partir de lui.

In [11]:
response = client.get("/openapi.json")
schema = response.json()

print(f"Title: {schema['info']['title']}")
print(f"Version: {schema['info']['version']}")
print(f"\nPaths:")
for path, methods in schema["paths"].items():
    for method in methods:
        if method in ("get", "post", "put", "delete"):
            print(f"  {method.upper():6s} {path}")


Title: Churn Predictor API
Version: 1.0.0

Paths:
  GET    /health
  GET    /model-info
  POST   /predict
  POST   /predict-batch


In [12]:
# Inspect the schema of one endpoint
import json
predict_schema = schema["paths"]["/predict"]["post"]
print(json.dumps({
    "summary": predict_schema.get("summary"),
    "request_body_ref": predict_schema["requestBody"]["content"]["application/json"]["schema"]["$ref"],
    "responses": list(predict_schema["responses"].keys()),
}, indent=2))


{
  "summary": "Predict",
  "request_body_ref": "#/components/schemas/ClientFeatures",
  "responses": [
    "200",
    "422"
  ]
}


💡 **Lecture pratique** : regardez les `responses` — vous devez voir au minimum `200` (succès) et `422` (validation Pydantic). Si `422` est absent, c'est que vous n'utilisez pas Pydantic correctement.

🧠 **Réflexe Tech Lead** : ce schéma est votre **contrat d'API**. Si vous le modifiez (ajout/suppression d'un champ obligatoire, changement de type), les clients qui en dépendent cassent. C'est pour ça qu'on **versionne** les APIs (`/v1/predict`, `/v2/predict`).

## 7. Lancer la suite pytest complète

Tout ce qu'on vient de faire dans le notebook est aussi dans `tests/test_api.py`. La différence : pytest **ré-exécute proprement** chaque test (fixture isolée, pas de pollution). C'est ce que tournera la CI.

In [13]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.13.0, typeguard-4.5.2, langsmith-0.8.9
collecting ... collected 10 items

tests/test_api.py::test_health PASSED                                    [ 10%]
tests/test_api.py::test_model_info PASSED                                [ 20%]
tests/test_api.py::test_predict_valid PASSED                             [ 30%]
tests/test_api.py::test_predict_missing_field PASSED                     [ 40%]
tests/test_api.py::test_predict_invalid_enum PASSED                      [ 50%]
tests/test_api.py::test_predict_negative_tenure PASSED                   [ 60%]
tests/test_api.py::test_predict_batch PASSED                             [ 70%]
tests/test_api.py::test_predict_batch_empty_rejected PASSED              [ 80%]
tests/test_api.py::test_root_redirects_to_docs PASSED            

In [14]:
# Properly close the lifespan
client.__exit__(None, None, None)
print("TestClient closed.")


TestClient closed.


## 8. Synthèse de la session

### 🎯 À remplir

> ## Synthèse S4 — Du modèle à l'API
>
> **API construite** :
> - Endpoints : `/health`, `/model-info`, `/predict`, `/predict-batch`
> - Validation Pydantic stricte sur 19 features
> - Modèle HGBT chargé au démarrage via lifespan (pas par requête)
> - 10 tests pytest passent (succès + 4 types d'erreurs en 422)
>
> **Réflexes Tech Lead retenus** :
> - 1. **Pydantic est ta première ligne de défense** : Literal pour les enums stricts, Field(ge=, le=) pour les bornes numériques. Sans ça, le modèle reçoit des inputs pourris et fait des prédictions farfelues.
> - 2. **Le lifespan FastAPI évite l'anti-pattern "modèle rechargé par requête"** : on passe d'une API à 1 seconde à une API à 1 ms. Critique en prod.
> - 3. **OpenAPI est le contrat d'API**, généré automatiquement par FastAPI. Versionner les routes (`/v1/`, `/v2/`) dès qu'on modifie le schéma pour ne pas casser les clients existants.
> - 4. **Une bonne API distingue les erreurs 422 (faute client) des 500 (faute serveur)**. Si on a un 500 sur un input invalide, c'est qu'on a oublié une validation Pydantic.
>
> **Ce qui reste pour S5** :
> - Monitoring : comment savoir si le modèle se dégrade en prod ?
> - Drift detection : nouvelles distributions, nouveaux schémas
> - Stratégie de retraining

---

### 🔭 Teaser Session 5 (19 juin)

> *"Aujourd'hui votre API tourne. Mais imaginez que demain, le marketing change la grille tarifaire et que tous les `MonthlyCharges` sont multipliés par 1.3. Votre modèle a été entraîné sur l'ancienne distribution — il va prédire faux sans vous prévenir. C'est le **drift**. Le 19 juin, on va apprendre à le détecter, à alerter, à décider quand retrainer. C'est la dernière brique : faire vivre un modèle dans le temps."*